# Fooocus_extend – VSW Google Colab

Robuste Schulungsfassung mit **isolierter Python-Umgebung**, Fortschrittsanzeige, direkter Civitai-Modellwahl und Fehlerdiagnose.

**Warum isoliert?** Fooocus_extend erhält unter `/content/fooocus_venv` einen eigenen Paketstand. Colabs vorinstallierte Torch-/CUDA-Pakete werden dadurch nicht mehr heruntergestuft oder überschrieben.

**Start:** Einstellungen wählen und auf die Start-Zelle klicken. Beim ersten Start werden sieben Phasen angezeigt. Sobald die Weboberfläche bereitsteht, endet der Fortschritts-Heartbeat automatisch und der öffentliche Link wird hervorgehoben.

Empfohlen: **Runtime 2025.07 · Python 3.11 · GPU/T4**.


In [ ]:

# @title ▶ Fooocus_extend STARTEN / NEUSTARTEN
import os, urllib.request

Fooocus_Profile = "realistic" #@param ["default", "realistic", "anime"]
Fooocus_Theme = "dark" #@param ["dark", "light"]
Tunnel = "gradio" #@param ["gradio", "cloudflared"]
Memory_patch = True #@param {type:"boolean"}
GoogleDrive_output = False #@param {type:"boolean"}
Use_latest_main = False #@param {type:"boolean"}
Force_rebuild_environment = False #@param {type:"boolean"}

# Optional: Civitai-Version-IDs oder direkte Download-URLs, mehrere Einträge mit ; trennen.
# Beispiel Checkpoint: 123456 oder 123456|mein_modell.safetensors
# Beispiel LoRA: 654321
Civitai_Checkpoints = "" #@param {type:"string"}
Civitai_LoRAs = "" #@param {type:"string"}
Use_Civitai_Secret = False #@param {type:"boolean"}

if Use_Civitai_Secret:
    try:
        from google.colab import userdata
        token = userdata.get('CIVITAI_TOKEN')
        if token:
            os.environ['CIVITAI_TOKEN'] = token
            print('Civitai-Token aus Colab Secrets geladen.')
    except Exception as e:
        raise RuntimeError("Colab Secret 'CIVITAI_TOKEN' konnte nicht geladen werden.") from e

url = "https://raw.githubusercontent.com/MlIelAst/Fooocus_Extend_VSW_Colab/main/vsw_launcher.py"
print("VSW-Launcher wird geladen …")
script = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")

replacements = {
    'PROFILE = "realistic"': f'PROFILE = {Fooocus_Profile!r}',
    'THEME = "dark"': f'THEME = {Fooocus_Theme!r}',
    'TUNNEL = "gradio"': f'TUNNEL = {Tunnel!r}',
    'MEMORY_PATCH = True': f'MEMORY_PATCH = {Memory_patch!r}',
    'GOOGLE_DRIVE_OUTPUT = False': f'GOOGLE_DRIVE_OUTPUT = {GoogleDrive_output!r}',
    'USE_LATEST_MAIN = False': f'USE_LATEST_MAIN = {Use_latest_main!r}',
    'FORCE_REBUILD = False': f'FORCE_REBUILD = {Force_rebuild_environment!r}',
    'EXTRA_CHECKPOINTS = ""': f'EXTRA_CHECKPOINTS = {Civitai_Checkpoints!r}',
    'EXTRA_LORAS = ""': f'EXTRA_LORAS = {Civitai_LoRAs!r}',
}
for old, new in replacements.items():
    if old not in script:
        raise RuntimeError(f"VSW-Launcher unerwartet geändert: Einstellung fehlt: {old}")
    script = script.replace(old, new, 1)

exec(compile(script, "vsw_launcher.py", "exec"), {"__name__": "__main__"})


## Modelle und LoRAs

Für den unkomplizierten Fooocus-Einsatz bevorzugt **SDXL-Checkpoints** und dazu passende **SDXL-LoRAs** verwenden. Eine Civitai-Modellseite kann mehrere Versionen enthalten; für den Direktdownload wird die **Version-ID** benötigt. Diese steht meist in der URL als `modelVersionId=...` oder hinter dem Download der konkreten Version.

- Checkpoints landen unter `/content/Fooocus_extend/models/checkpoints`.
- LoRAs landen unter `/content/Fooocus_extend/models/loras`.
- Mehrere Einträge mit `;` trennen.
- Optional kann mit `|dateiname.safetensors` ein eigener Dateiname gesetzt werden.
- Für zugriffsbeschränkte Downloads einen Civitai API-Token als **Colab Secret** `CIVITAI_TOKEN` speichern und `Use_Civitai_Secret` aktivieren. Den Token niemals in das Notebook oder GitHub schreiben.

Alternativ kann nach dem Start der in Fooocus_extend enthaltene **Civitai Helper** verwendet werden.

## Hinweise

- `Use_latest_main = False` für reproduzierbare Schulungen beibehalten.
- `GoogleDrive_output = False` ist der datensparsame Standard.
- Vollständiges Log bei Fehlern: `/content/fooocus_extend_startup.log`.
- Browser-Erweiterungsfehler entstehen clientseitig und sind im Colab-Launcher nicht immer sichtbar. Bei UI-Problemen deshalb zusätzlich Inkognito ohne Erweiterungen testen.
- Nach der Übung: **Laufzeit → Laufzeit trennen und löschen**.
